# nb159 — Boltz2 diagnostic with explicit file logging

Every step writes to /kaggle/working/diag.txt so we can pull and read after.
Minimal: install, validate import, run ONE compound.

In [ ]:
import os, subprocess, sys, time, json
from pathlib import Path

DIAG = Path('/kaggle/working/diag.txt')
def L(msg):
    with open(DIAG, 'a') as f:
        f.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    print(msg, flush=True)

L('=== nb159 START ===')
L(f'python: {sys.version}')
L(f'cwd: {os.getcwd()}')
import torch
L(f'torch: {torch.__version__}  cuda: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    L(f'  device: {torch.cuda.get_device_name(0)}  cc: {torch.cuda.get_device_capability(0)}')

In [ ]:
BZ_TARGET = '/kaggle/working/boltz_pkgs'
Path(BZ_TARGET).mkdir(exist_ok=True)
BOLTZ_BIN = f'{BZ_TARGET}/bin/boltz'

L(f'BOLTZ_BIN exists before install: {Path(BOLTZ_BIN).exists()}')
if not Path(BOLTZ_BIN).exists():
    L('Running pip install --target...')
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'pip', 'install',
                    '--target', BZ_TARGET, '-q',
                    'numpy==1.26.4', 'pandas==2.2.3', 'scipy==1.13.1',
                    'torch==2.4.0', 'torchmetrics==1.4.0', 'lightning==2.4.0',
                    'boltz', 'rdkit', 'pyyaml'], capture_output=True, text=True, timeout=1800)
    L(f'Install: rc={r.returncode}  elapsed={time.time()-t0:.0f}s')
    L(f'Install stderr tail (last 2000): {r.stderr[-2000:]}')
    bin_dir = Path(f'{BZ_TARGET}/bin')
    if bin_dir.exists():
        for f in bin_dir.iterdir(): f.chmod(0o755)
L(f'BOLTZ_BIN exists after install: {Path(BOLTZ_BIN).exists()}')
L(f'BOLTZ_BIN size: {Path(BOLTZ_BIN).stat().st_size if Path(BOLTZ_BIN).exists() else 0}')

In [ ]:
env = {**os.environ, 'PYTHONPATH': BZ_TARGET, 'PATH': f'{BZ_TARGET}/bin:' + os.environ.get('PATH','')}
L('Testing boltz CLI...')
r = subprocess.run([BOLTZ_BIN, '--help'], env=env, capture_output=True, text=True, timeout=60)
L(f'  boltz --help rc={r.returncode}')
L(f'  boltz --help stdout (first 600): {r.stdout[:600]}')
if r.returncode != 0:
    L(f'  boltz --help stderr (last 1000): {r.stderr[-1000:]}')

In [ ]:
PXR_SEQ = ('LDRRTVVPATQHVTGTAYIWYRSGLCEHHIVEAATRGNVMTPSCKLITEELLGRPVHIVQPVKAVCS'
           'IVKQSDCRPFNQRSFKKYFTMENKVMVLNQELIKLALNFKLQDGRPHGGIIYDLSGEEDPKSWIWE'
           'VLEAWDIKAQVGPVTYAVTSLPFLQLSQYLDQDLALYIHQAFRYGPNALLDLLTDTRKHADRLELN'
           'GLAIRLLPELEVALMLLTQHTLREEKAGNFETIAEPFNALVMQVMEGYREKDPEAKQNQELHIWAN'
           'KTKDPLLLEAHALDQFSCK')
RIFAMPICIN = 'CC1C(C(C(C=C(C(C(C=CC=C(C(C(C2=C(C3=C(C(=C2O)C)O)C(=O)C=C(N3)C)/C)O)C)OC(=O)C)C)O)C)O)C(=O)O1'
yaml_content = f'''version: 1
sequences:
- protein:
    id: A
    sequence: {PXR_SEQ}
- ligand:
    id: B
    smiles: {RIFAMPICIN}
properties:
- affinity:
    binder: B
'''
Path('/kaggle/working/rif.yaml').write_text(yaml_content)
L(f'Wrote rif.yaml ({len(yaml_content)} chars)')

OUT = Path('/kaggle/working/o_rif')
OUT.mkdir(exist_ok=True)
cmd = [BOLTZ_BIN, 'predict', '/kaggle/working/rif.yaml', '--out_dir', str(OUT),
       '--use_msa_server', '--diffusion_samples', '1', '--recycling_steps', '1', '--sampling_steps', '50']
L(f'CMD: {" ".join(cmd)}')
L('Running boltz predict (max 60 min)...')
t0 = time.time()
r = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=3600)
L(f'  rc={r.returncode}  elapsed={(time.time()-t0)/60:.1f}min')
L(f'  stdout tail (last 2000): {r.stdout[-2000:]}')
if r.returncode != 0:
    L(f'  stderr tail (last 2000): {r.stderr[-2000:]}')

# Look for outputs
all_files = list(OUT.rglob('*'))
L(f'Output dir files: {len(all_files)}')
for f in all_files[:20]:
    L(f'  {f.relative_to(OUT)}')
aff_files = list(OUT.rglob('*affinity*.json'))
L(f'Affinity files: {len(aff_files)}')
for jf in aff_files:
    try:
        d = json.load(open(jf))
        L(f'  {jf.name}: {d}')
    except Exception as e:
        L(f'  {jf.name}: ERR {e}')
L('=== nb159 END ===')